In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("IMDB Dataset.csv")
df.shape

(50000, 2)

In [3]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [5]:
df.drop_duplicates(inplace=True)

In [6]:
df.shape

(49582, 2)

# Text Pre-Processing

### 1. Convert to lowercase

In [7]:
df["review"] = df["review"].str.lower()

### 2. Remove URLs

In [8]:
import re

# sample_text = "abc is the word, abc" # abc => xyz

# new_text = re.sub("abc", "xyz", sample_text)

In [9]:
def remove_urls(text):
    texts = re.sub(r"http\S+", "", text) # (pattern, repl, string)
    return texts

df["review"] = df["review"].apply(remove_urls)

### 4. Remove HTML

In [10]:
def remove_html(text):
    texts = re.sub(r"<.*?>", "", text)
    return texts

df["review"] = df["review"].apply(remove_html)

### 3. Remove punctuations

In [11]:
def remove_punctuations(text):
    texts = re.sub(r"[^A-Za-z0-9\s]", "", text)
    return texts

df["review"] = df["review"].apply(remove_punctuations)

In [12]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production the filming tech...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


### 5. Remove Stopwords

In [13]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /home/austen-marwin-
[nltk_data]     gomes/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/austen-marwin-
[nltk_data]     gomes/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/austen-marwin-
[nltk_data]     gomes/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [14]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [15]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    filtered_tokens = []
    
    for word in tokens:
        if word not in stop_words:
            filtered_tokens.append(word)
            
    return " ".join(filtered_tokens)

df["review"] = df["review"].apply(remove_stopwords)

In [16]:
# def remove_stopwords(text):
#     tokens = word_tokenize(text)
#     stop_words = stopwords.words("english")

#     for word in tokens:
#         if word in stop_words:
#             text = text.replace(word, "") # Chat --> replace --> not good --> replace words inbetween the words
#     return text

# df["review"] = df["review"].apply(remove_stopwords)

In [17]:
df.head()

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production filming technique ...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically theres family little boy jake thinks...,negative
4,petter matteis love time money visually stunni...,positive


### 6. Stemming 

In [18]:
# running --> run

from nltk.stem import PorterStemmer

In [19]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [20]:
df.head()

,review,sentiment
0,one review mention watch 1 oz episod youll hoo...,positive
1,wonder littl product film techniqu unassum old...,positive
2,thought wonder way spend time hot summer weeke...,positive
3,basic there famili littl boy jake think there ...,negative
4,petter mattei love time money visual stun film...,positive


### 7. Encoding

In [21]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [22]:
y = df["sentiment"]

### 8. Vectorization

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

In [46]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4048094 stored elements and shape (49582, 5000)>

# Dataset & DataLoader

In [25]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [26]:
import torch
from torch.utils.data import DataLoader, TensorDataset

In [27]:
type(X_train) # Change to numpy array

scipy.sparse._csr.csr_matrix

In [28]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [29]:
y_test

29171    0
43589    1
38712    0
16045    0
5248     1
        ..
2923     1
15292    0
17849    0
38079    0
2691     0
Name: sentiment, Length: 9917, dtype: int64

In [30]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float(),
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float(),
)

In [32]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

# Build RNN

In [37]:
import torch.nn as nn
import torch.optim as optim

In [36]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN Layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional ==> shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0)
        # 1st value = hidden state of all the timesteps ==> 
        # 2nd value = final hidden state of last timesteps

        out = self.fc(out[:, -1, :]) # For every sentence in the batch, give me the hidden state produced at the final time step
        return out

In [38]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

# Train Model

In [39]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction --> output expexts 3D

        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size, ) => probability
        
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

    print(f"epoch = {epoch+1}/{epochs} & loss = {loss.item()}")

epoch = 1/10 & loss = 0.18282364308834076
epoch = 2/10 & loss = 0.20756949484348297
epoch = 3/10 & loss = 0.1393701434135437
epoch = 4/10 & loss = 0.15832358598709106
epoch = 5/10 & loss = 0.21024800837039948
epoch = 6/10 & loss = 0.4005480110645294
epoch = 7/10 & loss = 0.13508941233158112
epoch = 8/10 & loss = 0.26955893635749817
epoch = 9/10 & loss = 0.24304808676242828
epoch = 10/10 & loss = 0.24490395188331604


# Evaluate

In [48]:
model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0

    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)

        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 87.03236865987698
